# Phase 1: Meet the data

This notebook follows one real security flaw through the four sources CallGuard uses, and explains every field in plain words.

**You don't need to run anything.** Each grey box is code, and its result is saved right below it. To re-run it yourself, click **Run All** at the top and pick the `.venv` Python if VSCode asks.

Our example is **CVE-2024-41713**, a flaw in **Mitel MiCollab**. It's a good teacher because every source has something to say about it: attackers have used it, including in ransomware campaigns.

## Two ideas first

- **API:** an IVR for computers. You "call" a web address with a request in a fixed format, and it answers with data instead of a voice menu.
- **JSON:** the format of the answer. It's made of labeled fields, like a CRM record, and fields can hold lists and sub-records, like a ticket with notes attached.

In [1]:
import os
from pathlib import Path

import httpx
import pandas as pd
from dotenv import load_dotenv

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
load_dotenv(ROOT / ".env")
pd.set_option("display.max_colwidth", None)

CVE = "CVE-2024-41713"  # our example: a Mitel MiCollab flaw
NEW_CVE = "CVE-2026-20362"  # a brand-new Cisco Finesse flaw, for contrast later
http = httpx.Client(timeout=60, follow_redirects=True,
                    headers={"User-Agent": "callguard-notebook"})
NVD_HEADERS = {"apiKey": os.environ["NVD_API_KEY"]} if os.getenv("NVD_API_KEY") else {}

# Some fields name organizations by ID; these are the ones we meet today
SOURCE_NAMES = {
    "134c704f-9b21-4f2e-91b3-4a467353bcc0": "CISA (ADP)",
    "cve@mitre.org": "MITRE (CNA)",
    "psirt@cisco.com": "Cisco (CNA)",
    "nvd@nist.gov": "NVD",
}
print("Ready. Example CVE:", CVE)

Ready. Example CVE: CVE-2024-41713


## 1. NVD: the central case file

The **National Vulnerability Database** is run by NIST, a US government agency. Every published flaw gets a case file there, and NVD analysts enrich it: a severity score, the type of weakness, and the exact products and versions affected.

Think of it as the central case-management system. Here's the whole conversation with its API: one request, one answer.

In [2]:
r = http.get("https://services.nvd.nist.gov/rest/json/cves/2.0",
             params={"cveId": CVE}, headers=NVD_HEADERS)
print("We asked: ", r.request.url)
print("It said:  ", r.status_code, "(200 means OK)")
nvd = r.json()["vulnerabilities"][0]["cve"]
print("Fields in the record:", ", ".join(nvd))

We asked:  https://services.nvd.nist.gov/rest/json/cves/2.0?cveId=CVE-2024-41713
It said:   200 (200 means OK)
Fields in the record: id, sourceIdentifier, published, lastModified, vulnStatus, cveTags, descriptions, affected, metrics, cisaExploitAdd, cisaActionDue, cisaRequiredAction, cisaVulnerabilityName, weaknesses, configurations, references


### The main fields, in plain words

In [3]:
english = next(d["value"] for d in nvd["descriptions"] if d["lang"] == "en")
primary = next(m for m in nvd["metrics"]["cvssMetricV31"] if m["type"] == "Primary")["cvssData"]
cwe = nvd["weaknesses"][0]["description"][0]["value"]
pd.DataFrame([
    ("id", nvd["id"], "The public ticket number for this flaw"),
    ("published", nvd["published"][:10], "The day it went public"),
    ("lastModified", nvd["lastModified"][:10],
     "Last update. Phase 8 uses this to fetch only what changed"),
    ("vulnStatus", nvd["vulnStatus"],
     "NVD's work status. 'Analyzed' means an NVD analyst has reviewed it"),
    ("severity", f"{primary['baseScore']} {primary['baseSeverity']}",
     "CVSS score from 0 to 10, like ticket priority"),
    ("weakness", cwe, "The type of coding mistake. CWE-22 is 'path traversal'"),
    ("description", english, "What the flaw is"),
], columns=["field", "value", "what it means"])

,field,value,what it means
0,id,CVE-2024-41713,The public ticket number for this flaw
1,published,2024-10-21,The day it went public
2,lastModified,2026-08-04,Last update. Phase 8 uses this to fetch only what changed
3,vulnStatus,Analyzed,NVD's work status. 'Analyzed' means an NVD analyst has reviewed it
4,severity,9.1 CRITICAL,"CVSS score from 0 to 10, like ticket priority"
5,weakness,CWE-22,The type of coding mistake. CWE-22 is 'path traversal'
6,description,"A vulnerability in the NuPoint Unified Messaging (NPM) component of Mitel MiCollab through 9.8 SP1 FP2 (9.8.1.201) could allow an unauthenticated attacker to conduct a path traversal attack, due to insufficient input validation. A successful exploit could allow unauthorized access, enabling the attacker to view, corrupt, or delete users' data and system configurations.",What the flaw is


**The description in plain words:** in MiCollab up to version 9.8.1.201, someone who isn't logged in can read, change or delete files they shouldn't reach. They do it by sneaking `../` ("go up one folder") into a web request until they climb out of the folder they're allowed in. That trick is called *path traversal*.

You may notice the record also has a Spanish description. NVD translates descriptions, and CallGuard keeps only the English one.

### What makes it 9.1 out of 10?

The score is built from a few factors. Severity bands: **9.0–10 Critical**, 7.0–8.9 High, 4.0–6.9 Medium, 0.1–3.9 Low.

In [4]:
pd.DataFrame([
    ("Attack vector", primary["attackVector"], "Can be attacked remotely, over the network"),
    ("Attack complexity", primary["attackComplexity"], "No special conditions needed"),
    ("Privileges required", primary["privilegesRequired"], "The attacker doesn't need to log in"),
    ("User interaction", primary["userInteraction"], "Nobody has to click anything"),
    ("Confidentiality impact", primary["confidentialityImpact"], "The attacker can read data"),
    ("Integrity impact", primary["integrityImpact"], "The attacker can change or delete data"),
    ("Availability impact", primary["availabilityImpact"], "It doesn't take the system down"),
], columns=["factor", "value", "in plain words"])

,factor,value,in plain words
0,Attack vector,NETWORK,"Can be attacked remotely, over the network"
1,Attack complexity,LOW,No special conditions needed
2,Privileges required,NONE,The attacker doesn't need to log in
3,User interaction,NONE,Nobody has to click anything
4,Confidentiality impact,HIGH,The attacker can read data
5,Integrity impact,HIGH,The attacker can change or delete data
6,Availability impact,NONE,It doesn't take the system down


### Who scored it?

NVD keeps its own score (**Primary**) next to other organizations' scores (**Secondary**). The last line is a different kind of rating, SSVC, which section 2 explains.

In [5]:
for kind, items in nvd["metrics"].items():
    for m in items:
        score = m.get("cvssData", {}).get("baseScore", "-")
        who = SOURCE_NAMES.get(m["source"], m["source"])
        print(f"{kind:14} {m.get('type', ''):10} score {score:>4}   by {who}")

cvssMetricV31  Primary    score  9.1   by NVD
cvssMetricV31  Secondary  score  9.1   by CISA (ADP)
ssvcV203                  score    -   by CISA (ADP)


### Which products and versions? NVD's analysis (CPE)

NVD describes affected products with **CPE** names. Read `cpe:2.3:a:mitel:micollab:*` as: **a**pplication (`o` would mean operating system, `h` hardware), vendor **mitel**, product **micollab**, any version (`*`), narrowed by a version range. It's like the "affected models and firmware" table in a vendor field notice.

In [6]:
rows = []
for conf in nvd.get("configurations", []):
    for node in conf["nodes"]:
        for m in node["cpeMatch"]:
            rows.append({
                "product (CPE name)": m["criteria"],
                "from version": m.get("versionStartIncluding") or m.get("versionStartExcluding") or "(any)",
                "up to version": m.get("versionEndIncluding") or m.get("versionEndExcluding") or "(any)",
                "including that version?": ("yes" if "versionEndIncluding" in m
                                            else "no" if "versionEndExcluding" in m else "-"),
            })
pd.DataFrame(rows)

,product (CPE name),from version,up to version,including that version?
0,cpe:2.3:a:mitel:micollab:*:*:*:*:*:-:*:*,(any),9.8.1.201,yes


### The vendor's own version list (`affected`)

NVD also copies the version list from CVE.org (section 2) into a field called `affected`. For this CVE, the organization that filed it (MITRE) left the product as "n/a", and CISA filled it in. This field matters most for brand-new CVEs that NVD hasn't analyzed yet (section 5).

In [7]:
rows = []
for a in nvd.get("affected", []):
    for ad in a["affectedData"]:
        for v in ad.get("versions", []):
            rows.append({
                "supplied by": SOURCE_NAMES.get(a["source"], a["source"]),
                "vendor": ad.get("vendor"),
                "product": ad.get("product"),
                "version": v.get("version"),
                "up to (including)": v.get("lessThanOrEqual", ""),
                "below": v.get("lessThan", ""),
            })
pd.DataFrame(rows)

,supplied by,vendor,product,version,up to (including),below
0,MITRE (CNA),n/a,n/a,n/a,,
1,CISA (ADP),mitel,micollab,0,9.8.1.201,


### "Is it being exploited?" NVD copies CISA's answer

When CISA confirms attackers are using a flaw (section 3), NVD copies that into the record. It doesn't copy the ransomware flag, though; that lives only in CISA's list.

In [8]:
pd.DataFrame([(k, nvd.get(k)) for k in ("cisaVulnerabilityName", "cisaExploitAdd",
                                        "cisaActionDue", "cisaRequiredAction")],
             columns=["field", "value"])

,field,value
0,cisaVulnerabilityName,Mitel MiCollab Path Traversal Vulnerability
1,cisaExploitAdd,2025-01-07
2,cisaActionDue,2025-01-28
3,cisaRequiredAction,Apply mitigations per vendor instructions or discontinue use of the product if mitigations are unavailable.


### Where's the fix?

References are links, tagged by type. The **Vendor Advisory** is where an engineer finds the fix: here, Mitel's advisory MISA-2024-0029.

In [9]:
pd.DataFrame([(ref["url"], ", ".join(ref.get("tags", []))) for ref in nvd["references"]],
             columns=["link", "tags"])

,link,tags
0,https://www.mitel.com/support/security-advisories/mitel-product-security-advisory-misa-2024-0029,Vendor Advisory
1,https://www.cisa.gov/known-exploited-vulnerabilities-catalog?field_cve=CVE-2024-41713,US Government Resource


## 2. CVE.org: the original ticket

The **CVE Program** issues the CVE numbers. Each CVE is filed by a **CNA** (CVE Numbering Authority). That's often the vendor itself (Cisco files its own), and sometimes a coordinator such as MITRE or GitHub. Later, an **ADP** (Authorized Data Publisher), mainly CISA, can add information.

So CVE.org holds the original ticket as filed, plus CISA's notes. NVD is a second team that copies all of that and adds its own analysis.

In [10]:
rec = http.get(f"https://cveawg.mitre.org/api/cve/{CVE}").json()
meta, cna = rec["cveMetadata"], rec["containers"]["cna"]
print("Filed by (the CNA):", meta["assignerShortName"])
print("ID reserved:", meta["dateReserved"][:10], "| made public:", meta["datePublished"][:10])
print("CNA's product data:", [(a.get("vendor"), a.get("product")) for a in cna.get("affected", [])])
print("CNA's links:", [ref["url"] for ref in cna.get("references", [])])
for adp in rec["containers"].get("adp", []):
    print(f"\nAdded later by {adp['providerMetadata']['shortName']} ({adp.get('title', '')}):")
    for m in adp.get("metrics", []):
        if "cvssV3_1" in m:
            print("  severity:", m["cvssV3_1"]["baseScore"], m["cvssV3_1"]["baseSeverity"])
        elif m.get("other", {}).get("type") == "ssvc":
            options = m["other"]["content"]["options"]
            print("  SSVC triage:", ", ".join(f"{k}: {v}" for o in options for k, v in o.items()))
        elif m.get("other", {}).get("type") == "kev":
            print("  in CISA KEV since:", m["other"]["content"]["dateAdded"])
    for a in adp.get("affected", []):
        for v in a.get("versions", []):
            print(f"  affected: {a['vendor']} {a['product']}, versions up to "
                  f"{v.get('lessThanOrEqual') or v.get('lessThan')}")
    for p in adp.get("problemTypes", []):
        for d in p["descriptions"]:
            print("  weakness:", d.get("description"))

Filed by (the CNA): mitre
ID reserved: 2024-07-22 | made public: 2024-10-21
CNA's product data: [('n/a', 'n/a')]
CNA's links: ['https://www.mitel.com/support/security-advisories/mitel-product-security-advisory-misa-2024-0029']

Added later by CISA-ADP (CISA ADP Vulnrichment):
  severity: 9.1 CRITICAL
  SSVC triage: Exploitation: active, Automatable: yes, Technical Impact: total
  in CISA KEV since: 2025-01-07
  affected: mitel micollab, versions up to 9.8.1.201
  weakness: CWE-22 Improper Limitation of a Pathname to a Restricted Directory ('Path Traversal')


**What to notice:**

- The ID was **reserved in July 2024** but made **public in October 2024**. That gap is normal: it gives the vendor time to build a fix before attackers learn about the flaw.
- MITRE filed the record with almost no detail ("n/a"). **CISA added** the severity, the weakness type and the affected versions.
- **SSVC** is CISA's quick triage, like tags on an incident:
  - *Exploitation: active*: attackers are using it now.
  - *Automatable: yes*: attacks can be scripted to hit many systems at once.
  - *Technical impact: total*: the attacker gets full control of the affected part.

## 3. CISA KEV: the "confirmed exploited" list

The **Known Exploited Vulnerabilities** catalog is CISA's list of flaws that attackers are confirmed to be using. Think of it as the major-incident board: a flaw only goes up there with evidence.

In [11]:
kev = http.get("https://www.cisa.gov/sites/default/files/feeds/"
               "known_exploited_vulnerabilities.json").json()
print(f"KEV catalog {kev['catalogVersion']}: {len(kev['vulnerabilities'])} flaws confirmed exploited")
entry = next(v for v in kev["vulnerabilities"] if v["cveID"] == CVE)
pd.DataFrame(list(entry.items()), columns=["field", "value"])

KEV catalog 2026.10.16: 1748 flaws confirmed exploited


,field,value
0,cveID,CVE-2024-41713
1,vendorProject,Mitel
2,product,MiCollab
3,vulnerabilityName,Mitel MiCollab Path Traversal Vulnerability
4,dateAdded,2025-01-07
5,shortDescription,"Mitel MiCollab contains a path traversal vulnerability that could allow an attacker to gain unauthorized and unauthenticated access. This vulnerability can be chained with CVE-2024-55550, which allows an unauthenticated, remote attacker to read arbitrary files on the server."
6,requiredAction,Apply mitigations per vendor instructions or discontinue use of the product if mitigations are unavailable.
7,dueDate,2025-01-28
8,knownRansomwareCampaignUse,Known
9,forensicTriage,No


**The fields that matter most:**

- **dateAdded:** when CISA confirmed exploitation.
- **dueDate:** the deadline for US federal agencies to fix it. It's a useful urgency signal for everyone else too.
- **requiredAction:** what to do, in one line.
- **knownRansomwareCampaignUse:** "Known" means ransomware gangs have used this flaw. "Unknown" means CISA hasn't confirmed it, which is not the same as "no". This field exists **only** in KEV, which is why CallGuard still reads KEV directly.
- **forensicTriage:** "Yes" means CISA tells US federal agencies to check for signs of a break-in, not just patch (under directive BOD 26-04). It's "No" for this flaw; about 80 of the roughly 1,750 KEV entries say "Yes".

Also notice the short description: attackers can **chain** this flaw with a second one, CVE-2024-55550, which is in KEV too. Questions like "which flaws get used together?" are the multi-step kind we'll write in Phase 3.

## 4. FIRST EPSS: the forecast

The **Exploit Prediction Scoring System** estimates the chance that attackers will try a flaw in the **next 30 days**. Like a call-volume forecast, it predicts rather than confirms; KEV is what confirms.

Here's our MiCollab flaw next to a brand-new Cisco Finesse flaw.

In [12]:
scores = http.get("https://api.first.org/data/v1/epss",
                  params={"cve": f"{CVE},{NEW_CVE}"}).json()["data"]
pd.DataFrame([{
    "CVE": s["cve"],
    "chance of exploitation in the next 30 days": f"{float(s['epss']):.1%}",
    "riskier than this share of all CVEs": f"{float(s['percentile']):.1%}",
    "scored on": s["date"],
} for s in scores])

,CVE,chance of exploitation in the next 30 days,riskier than this share of all CVEs,scored on
0,CVE-2026-20362,0.5%,39.3%,2026-10-16
1,CVE-2024-41713,97.2%,99.9%,2026-10-16


The MiCollab flaw is near the very top of all CVEs. The new Finesse flaw is low for now, but scores are updated daily, so this can change.

## 5. Why NVD often lacks version data, and what fills the gap

NVD analysts review CVEs after they're published, and they've had a large backlog since 2024. Our contrast CVE, the Cisco Finesse flaw, was published on 7 October 2026. Let's see what NVD has.

In [13]:
new = http.get("https://services.nvd.nist.gov/rest/json/cves/2.0",
               params={"cveId": NEW_CVE}, headers=NVD_HEADERS).json()["vulnerabilities"][0]["cve"]
print(NEW_CVE, "| published", new["published"][:10], "| NVD status:", new["vulnStatus"])
print("NVD's own product analysis (CPE):", "none yet" if not new.get("configurations") else "present")
for a in new.get("affected", []):
    for ad in a["affectedData"]:
        versions = [v["version"] for v in ad.get("versions", [])]
        print(f"\nVendor's list, from {SOURCE_NAMES.get(a['source'], a['source'])}: "
              f"{ad['vendor']} / {ad['product']}, {len(versions)} affected versions:")
        print("  " + ", ".join(versions))

CVE-2026-20362 | published 2026-10-07 | NVD status: Awaiting Analysis
NVD's own product analysis (CPE): none yet

Vendor's list, from Cisco (CNA): Cisco / Cisco Finesse, 28 affected versions:
  12.6(1), 12.6(1)ES1, 12.6(1)ES2, 12.6(1)ES3, 12.6(1)ES4, 12.6(1)ES5, 12.6(1)ES6, 12.6(1)ES7, 12.6(1)ES7_ET, 12.6(2), 12.6(1)ES8, 12.6(1)ES9, 12.6(2)ES1, 12.6(1)ES10, 12.6(1)ES11, 12.6(2)ES2, 12.6(2)ES3, 12.6(2)ES4, 12.6(2)ES5, 15.0(1), 12.6(2)ES6, 15.0(1)ES202508, 15.0(1)ES202511, 15.0(1)ES202602, 15.0(1)SU1, 12.6(2)ES7, 15.0(1)SU2, 12.6(2)ES8


NVD hasn't analyzed it yet, but **Cisco, as the CNA, listed every affected version**, and NVD already carries that list in `affected`.

Across our ~880 in-scope CVEs (see the data check in NOTES.md), for those published in 2025–26:

| Version data from | Coverage |
|---|---|
| NVD's own analysis (CPE) | 54% |
| The vendor's list (`affected`) | 84% |
| **Either one** | **89%** |

For older CVEs, coverage is 100%. So CallGuard reads versions from **both** fields, and a separate CVE.org download isn't needed.

Notice the version names, too: `12.6(1)ES1` is not a simple number. That's why Phase 5 checks "is my version affected?" with plain code rather than leaving it to the AI.

## 6. One CVE, four views: who provides what

| What we need | NVD | CVE.org | CISA KEV | FIRST EPSS |
|---|---|---|---|---|
| Description | ✓ (copied) | ✓ the original | a short one | – |
| Severity (CVSS) | ✓ its own and others' | from CISA | – | – |
| Weakness type (CWE) | ✓ | from CISA | ✓ | – |
| Affected products and versions | ✓ its analysis (CPE) and the vendor's list (`affected`) | ✓ the vendor's list | product name only | – |
| Exploited in the wild? | ✓ copied from KEV | ✓ CISA's triage | ✓ **the source of truth** | – |
| Used by ransomware? | – | – | ✓ **only here** | – |
| Check for signs of a break-in? | – | – | ✓ **only here** | – |
| Chance of exploitation soon | – | – | – | ✓ **only here** |
| Where's the fix? | ✓ tagged links | ✓ links | required action | – |

**What this means for CallGuard:** NVD is the main source, since it already bundles CVE.org's data and KEV's "exploited" flag. KEV adds the ransomware flag and same-day updates. EPSS adds the forecast. We used CVE.org today to understand where the data comes from, and we'll keep it as a backup.

**Good to know:** NVD's terms ask apps that use its API to show this notice: *"This product uses the NVD API but is not endorsed or certified by the NVD."* We'll add it to the chat app in Phase 10.

## Next

We'll choose which products CallGuard covers. That's your call, as the contact-center expert.